# Lab 16-11 · From Algorithms to Datapaths

Five parts, each built on `hwarch.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math, os, subprocess, tempfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import hwarch as hw
from make_vectors import angles, build

## Part 1 · Hardware budgets

*Predict:* DSP blocks for a 64-tap symmetric filter at 50 MS/s on 400 MHz blocks.

In [2]:
print("taps   fs(MS/s)  blocks@400MHz  blocks@600MHz  (folded)")
for taps in (32, 64, 128, 256):
    for fs in (25e6, 50e6, 100e6, 200e6):
        m = taps // 2
        print(f"{taps:4d}   {fs / 1e6:6.0f}    {hw.units_needed(m, fs, 400e6):8d}       {hw.units_needed(m, fs, 600e6):8d}")
assert hw.units_needed(32, 50e6, 400e6) == 4

taps   fs(MS/s)  blocks@400MHz  blocks@600MHz  (folded)
  32       25           1              1
  32       50           2              2
  32      100           4              3
  32      200           8              6
  64       25           2              2
  64       50           4              3
  64      100           8              6
  64      200          16             11
 128       25           4              3
 128       50           8              6
 128      100          16             11
 128      200          32             22
 256       25           8              6
 256       50          16             11
 256      100          32             22
 256      200          64             43


## Part 2 · Look-ahead

*Predict:* the largest difference between the original and look-ahead outputs.

In [3]:
rng = np.random.default_rng(11)
x = rng.standard_normal(2000)
y1 = hw.iir1(0.9, x)
y2 = hw.iir1_lookahead2(0.9, x)
y4 = np.zeros_like(x)
for n in range(len(x)):
    y4[n] = 0.9 ** 4 * (y4[n - 4] if n >= 4 else 0) + sum(0.9 ** k * (x[n - k] if n >= k else 0) for k in range(4))
print("max |y - y_LA2| =", np.max(np.abs(y1 - y2)), " max |y - y_LA4| =", np.max(np.abs(y1 - y4)))
assert np.max(np.abs(y1 - y2)) < 1e-10 and np.max(np.abs(y1 - y4)) < 1e-10
print("iteration bounds (ns): original", hw.iteration_bound([(2.5, 1)]), " LA2", hw.iteration_bound([(2.5, 2)]), " LA4", hw.iteration_bound([(2.5, 4)]))

max |y - y_LA2| = 1.7763568394002505e-15  max |y - y_LA4| = 2.220446049250313e-15
iteration bounds (ns): original 2.5  LA2 1.25  LA4 0.625


## Part 3 · Shift-and-add constants

*Predict:* which coefficients save adders in CSD form.

In [4]:
half_band = [0.5, 0.3125, -0.09375, 0.046875, -0.0234375]
tot_csd = tot_bin = 0
for c in half_band:
    n = int(round(abs(c) * 2 ** 8))
    digits = hw.csd(abs(c), 8)
    assert abs(sum(s * 2.0 ** e for s, e in digits) - abs(c)) < 1e-12
    b = bin(n).count("1") - 1
    tot_csd += hw.csd_adders(abs(c), 8); tot_bin += b
    print(f"{c:+.7f}: CSD {digits} -> {hw.csd_adders(abs(c), 8)} adders; binary {b}")
print("total adders: CSD", tot_csd, "binary", tot_bin)
assert hw.csd_adders(0.875, 4) == 1 and hw.csd_adders(0.9375, 4) == 1

+0.5000000: CSD [(1, -1)] -> 0 adders; binary 0
+0.3125000: CSD [(1, -4), (1, -2)] -> 1 adders; binary 1
-0.0937500: CSD [(-1, -5), (1, -3)] -> 1 adders; binary 1
+0.0468750: CSD [(-1, -6), (1, -4)] -> 1 adders; binary 1
-0.0234375: CSD [(-1, -7), (1, -5)] -> 1 adders; binary 1
total adders: CSD 4 binary 4


## Part 4 · CORDIC convergence

*Predict:* the number of iterations for a worst-case error below $10^{-4}$.

In [5]:
th = np.linspace(-math.pi / 2, math.pi / 2, 2001)
ns = list(range(6, 21))
worst = []
for n in ns:
    worst.append(max(max(abs(hw.cordic_rotate(t, n)[0] - math.cos(t)), abs(hw.cordic_rotate(t, n)[1] - math.sin(t))) for t in th))
need = next(n for n, w in zip(ns, worst) if w < 1e-4)
print("iterations needed for 1e-4:", need)
assert need == 15
plt.figure(figsize=(5, 3)); plt.semilogy(ns, worst, "o-"); plt.xlabel("iterations"); plt.ylabel("worst error"); plt.show()
m, a = hw.cordic_vector(3.0, 4.0, 16)
print(f"vectoring (3, 4): magnitude {m:.4f}, angle {math.degrees(a):.2f} deg")
assert abs(m - 5) < 1e-4 and abs(math.degrees(a) - 53.13) < 0.005

iterations needed for 1e-4: 15
vectoring (3, 4): magnitude 5.0000, angle 53.13 deg


## Part 5 · The hardware

*Predict:* the worst fixed-point error in LSB over the 64 test angles.

In [6]:
assert open("vectors.hex").read().split() == build(), "vectors.hex is stale: rerun make_vectors.py"
errs = []
for t in angles:
    xq, yq, _ = hw.cordic_fixed(t, 14)
    a = t / 2 ** 15 * math.pi
    errs += [abs(xq - math.cos(a) * 2 ** 14), abs(yq - math.sin(a) * 2 ** 14)]
print(f"worst fixed-point error: {max(errs):.2f} LSB")
assert abs(max(errs) - 6.86) < 0.005
try:
    vvp = os.path.join(tempfile.mkdtemp(), "cordic.vvp")
    subprocess.run(["iverilog", "-g2012", "-o", vvp, "cordic_tb.v", "cordic.v"], check=True, capture_output=True)
    out = subprocess.run(["vvp", vvp], capture_output=True, text=True).stdout
    print([ln for ln in out.splitlines() if "TEST" in ln][0]); assert "TEST PASSED" in out
except FileNotFoundError:
    print("Icarus Verilog not installed here; CI runs the testbench (tools/run_hdl.py).")

worst fixed-point error: 6.86 LSB
TEST PASSED (64 angles, bit-exact)
